# Segmentação Automática da Fatia Sagital Média do Corpo Caloso

Fluxo do notebook:
1. Explorar os dados 3D do HCP
2. Préprocessamento: extrair a fatia sagital média e salvar dados para treinamento
3. Treinar uma rede UNet com PyTorch Lightning
4. Avaliar predições e métricas

Execute as células **em ordem**. Funções e classes auxiliares estão no arquivo **brainhack.py**.


## Imports e Configuração
Códigos utilitários para instalar biblioteca MONAI e importar módulos externos.

In [ ]:
!pip install monai
import sys
from pathlib import Path

def add_brainhack_to_path():
    """Acha brainhack.py venha ele como Dataset (/kaggle/input) ou Utility Script (/kaggle/usr/lib)."""
    for root in (Path("/kaggle/input"), Path("/kaggle/usr/lib"), Path.cwd()):
        for p in root.rglob("brainhack.py") if root.exists() else ():
            sys.path.insert(0, str(p.parent))
            return p.parent
    raise FileNotFoundError("brainhack.py não encontrado: anexe o dataset ou o utility script")

print(add_brainhack_to_path())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.1 MB/s eta 0:00:00
/kaggle/input/competitions/sao-paulo-alberta-brain-hack-3-0


In [ ]:
import os
import json
import random
import nibabel as nib
from collections import defaultdict
from glob import glob
from math import nan
from pathlib import Path

import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import pytorch_lightning as pl
import SimpleITK as sitk
import torch
import torch.nn as nn
import torch.nn.functional as F
try:
    import torchvision          # só usado na visualização com make_grid
except ModuleNotFoundError:     # no Kaggle já vem instalado; localmente é opcional
    torchvision = None
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from torch import Tensor
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# Leitura de Arquivos .nii.gz
Se tudo deu certo na configuração, os arquivos estão em DATA_DIR

In [ ]:
# Caminhos do Kaggle. Rodando localmente (fora do Kaggle) eles não existem,
# então caímos para a pasta do próprio notebook -- que é onde ficam tanto o
# dataset quanto os train/val/test.json.
KAGGLE_COMP = Path("/kaggle/input/competitions/sao-paulo-alberta-brain-hack-3-0")
EM_KAGGLE = KAGGLE_COMP.exists()

if EM_KAGGLE:
    DATA_ROOT = Path("/kaggle/working")
    COMP_DIR = KAGGLE_COMP
    DATA_JSON = KAGGLE_COMP
else:
    DATA_ROOT = Path.cwd()
    COMP_DIR = Path.cwd()
    DATA_JSON = Path.cwd()

print("Kaggle" if EM_KAGGLE else f"Local: {COMP_DIR}")

SAGITTAL_AXIS = 0

# Entrada da rede: "fa" (1 canal, como no notebook original) ou "tensor"
# (6 canais, as componentes unicas do tensor de difusao D).
INPUT_MODE = "tensor"
N_IN = {"fa": 1, "tensor": 6}[INPUT_MODE]

# Pasta separada por tipo de entrada: os .npz tem o mesmo nome por sujeito,
# entao compartilhar a pasta faria o treino ler silenciosamente as fatias da
# entrada anterior.
PROCESSED_DATA_FOLDER = f"preprocessed_cc_{INPUT_MODE}"

print(f"Entrada: {INPUT_MODE} ({N_IN} canais) -> {PROCESSED_DATA_FOLDER}")

# Pastas que nunca contêm sujeitos e que têm MUITOS arquivos: percorrê-las
# deixa a busca lenta quando COMP_DIR é a raiz do repositório.
IGNORAR = {".git", ".venv", "venv", "env", "__pycache__", "logs", "node_modules"}


def find_data_root(start: Path) -> Path:
    """Retorna o diretorio pai das pastas de sujeito, a qualquer profundidade.

    Identifica sujeito pelo conteudo (presenca dos NIfTI esperados),
    nao por profundidade nem por formato do nome.
    """
    targets = {"FA.nii", "evals.nii", "evecs.nii", "cc_mask_mricloud_1.25.nii"}
    # os.walk em vez de rglob("*") para poder PODAR a árvore: o os.walk também
    # para no primeiro acerto, enquanto o rglob lista tudo antes de testar.
    for root, dirs, files in os.walk(start):
        dirs[:] = sorted(d for d in dirs if d not in IGNORAR and not d.startswith("."))
        if targets <= set(files):
            return Path(root).parent
    raise FileNotFoundError(
        f"Nenhuma pasta contendo {sorted(targets)} encontrada em {start}"
    )

DATA_DIR = find_data_root(
    COMP_DIR,
)

print(DATA_DIR)

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset não encontrado: {DATA_DIR.resolve()}")

for _split in ("train", "val", "test"):
    if not (DATA_JSON / f"{_split}.json").exists():
        raise FileNotFoundError(f"Split não encontrado: {DATA_JSON / f'{_split}.json'}")


def load_nifti(path: Path) -> np.ndarray:
    return nib.load(path).get_fdata()


In [ ]:
!ls "{DATA_DIR}"

102109	125424	146735	176845	206727	299760	421226	558657	728454	886674
102614	126426	146836	177140	206828	300719	453542	558960	757764	902242
102715	127832	147636	180230	206929	314225	454140	559457	763557	905147
103212	130518	151324	186545	210112	325129	461743	561949	765864	911849
106824	130720	151930	186848	211619	342129	463040	567759	774663	933253
108020	135124	152225	188145	211821	349244	468050	589567	788674	962058
111211	135629	152427	191235	213017	350330	481042	590047	814548	970764
113316	136631	153126	192237	213522	360030	510225	634748	815247	987074
115724	137532	161832	193845	219231	368753	513130	635245	818455	989987
117021	138130	165436	194443	227533	376247	516742	654552	825553
118831	138332	165941	198047	238033	378756	518746	675661	825654
119025	139435	167440	199352	255740	392447	519647	680452	828862
120414	143224	168947	200513	257946	394956	541640	692964	832651
123723	144933	169545	206323	274542	413934	552241	694362	869472
125222	145632	175136	206525	281135	419239	555954	698168

In [ ]:
!ls "{DATA_DIR}/102109"

AD.nii			   FA.nii		 T1_1.25.nii
cc_mask_fs_1.25.nii	   MD.nii		 T1_brain_1.25.nii
cc_mask_mricloud_1.25.nii  mean_b0.nii		 T1_brain_mask_1.25.nii
evals.nii		   nodif_brain_mask.nii
evecs.nii		   RD.nii


In [ ]:
evecs_img = nib.load(f"{DATA_DIR}/102109/evecs.nii")
evals_img = nib.load(f"{DATA_DIR}/102109/evals.nii")

evecs = evecs_img.get_fdata(dtype=np.float32)   # (X, Y, Z, 3, 3)
evals = evals_img.get_fdata(dtype=np.float32)   # (X, Y, Z, 3)

if evecs.shape[-1] == 9:                        # achatado (X, Y, Z, 9)
    evecs = evecs.reshape(*evecs.shape[:-1], 3, 3)

# D = V diag(lambda) V^T -- nao basta multiplicar evals por evecs.
#
# Os autovetores estao nas COLUNAS de V: evecs[..., :, i] corresponde a
# evals[..., i]. E por isso que o ultimo termo do einsum e '...kj' (a
# transposta) e nao '...jk'. Trocar os dois produz uma matriz simetrica de
# aparencia plausivel, com os mesmos autovalores, mas ERRADA: neste dataset
# as duas convencoes diferem por ~40 graus na direcao principal.
D = np.einsum('...ij,...j,...kj->...ik', evecs, evals, evecs)   # (X, Y, Z, 3, 3)

# Sanidade: D tem que ser simetrica e o traco tem que bater com a soma dos evals.
print("D:", D.shape)
print("simetrica?          ", np.allclose(D, np.swapaxes(D, -1, -2)))
print("traco == soma evals?",
      np.allclose(np.trace(D, axis1=-2, axis2=-1), evals.sum(-1), atol=1e-6))


## O tensor como entrada da rede: 6 canais, e cuidado com o sinal

Duas decisoes que valem ser explicitas antes de mexer no dataset:

**1. 6 canais, nao 9.** `D` e simetrica, entao as 9 componentes carregam apenas
6 numeros independentes (`Dxx Dxy Dxz Dyy Dyz Dzz`). Passar as 9 so daria a
rede canais duplicados.

**2. Nao clipar em zero.** As componentes de fora da diagonal (`Dxy`, `Dxz`,
`Dyz`) sao legitimamente **negativas** — cerca de 6% dos valores neste dataset.
Um `np.clip(D, a_min=0, ...)` zera todas elas e joga fora exatamente a
informacao de orientacao que e o motivo de usar `D` em vez da FA. A escala
tambem importa: `0.01 mm²/s` fica ~3x acima da difusao livre da agua, entao
dividir por esse valor comprime tudo em `[0, 0.3]` e desperdica boa parte da
faixa dinamica. A normalizacao usada aqui esta em `ScaleDiffusivity`.


In [ ]:
# ---------------------------------------------------------------------------
# Utilidades para usar o tensor de difusao como entrada
# ---------------------------------------------------------------------------

# As 6 componentes unicas de uma matriz simetrica, na ordem em que viram
# canais: Dxx Dxy Dxz Dyy Dyz Dzz. Listas (e nao np.triu_indices) porque
# servem para indexar tanto arrays numpy quanto tensores torch.
TRIU = ([0, 0, 0, 1, 1, 2], [0, 1, 2, 1, 2, 2])
N_TENSOR_CHANNELS = 6

# Difusividade de referencia (mm^2/s): a da agua livre a 37 C. Escala FIXA
# usada para normalizar o tensor -- ver ScaleDiffusivity.
D_REF = 3e-3


def tensor_to_channels(D):
    """(..., 3, 3) -> (..., 6): as componentes unicas da tensor simetrica."""
    return D[..., TRIU[0], TRIU[1]]


def channels_to_tensor(comp):
    """(..., 6) -> (..., 3, 3): reconstroi a matriz simetrica completa."""
    D = np.empty(comp.shape[:-1] + (3, 3), dtype=comp.dtype)
    D[..., TRIU[0], TRIU[1]] = comp
    D[..., TRIU[1], TRIU[0]] = comp
    return D


def fa_from_channels(comp):
    """FA a partir das 6 componentes, sem autodecomposicao.

    FA^2 = 3/2 * ||D - (tr D / 3) I||^2 / ||D||^2, e as duas normas de
    Frobenius saem direto das componentes. Confere com o FA.nii do dataset
    ate ~1e-5. E invariante a escala, entao da o mesmo resultado antes ou
    depois da normalizacao.
    """
    dxx, dxy, dxz, dyy, dyz, dzz = (comp[..., k] for k in range(6))
    tr = dxx + dyy + dzz
    norm2 = dxx**2 + dyy**2 + dzz**2 + 2 * (dxy**2 + dxz**2 + dyz**2)
    with np.errstate(invalid="ignore", divide="ignore"):
        fa = np.sqrt(np.clip(1.5 * (1.0 - (tr**2 / 3.0) / norm2), 0.0, 1.0))
    return np.nan_to_num(fa)


def md_from_channels(comp):
    """Difusividade media = traco / 3 (usada so para visualizar)."""
    return (comp[..., 0] + comp[..., 3] + comp[..., 5]) / 3.0


def rotation_matrix_x(angle_rad):
    """Rotacao em torno do eixo sagital (eixo 0) = o plano (y, z) da fatia."""
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]], dtype=np.float32)


def rotate_tensor_channels(comp, R):
    """Gira um campo tensorial: D' = R D R^T em cada voxel. `comp` e (..., 6).

    Isso NAO e opcional quando se gira a imagem. Um campo escalar (FA) so
    precisa que o grid gire; num campo tensorial as componentes precisam
    girar junto, senao a orientacao codificada em D deixa de corresponder a
    anatomia que aparece na imagem.
    """
    D = channels_to_tensor(np.asarray(comp, dtype=np.float32))
    D = np.einsum('ij,...jk,lk->...il', R, D, R)
    return np.ascontiguousarray(tensor_to_channels(D), dtype=np.float32)


In [ ]:
array = D

In [ ]:
array.shape

(145, 174, 145)

In [ ]:
array.min(), array.max()

(np.float64(-7.879262924194336), np.float64(3366.887939453125))

In [ ]:
# As 6 componentes numa fatia sagital, mais MD e FA derivadas delas.
comp = tensor_to_channels(D)          # (X, Y, Z, 6)
nomes = ["Dxx", "Dxy", "Dxz", "Dyy", "Dyz", "Dzz"]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for k, ax in enumerate(axes.flat[:6]):
    sl = comp[100, :, :, k].T
    if k in (0, 3, 5):   # diagonal: sempre positiva
        ax.imshow(sl, cmap="gray", origin="lower", vmin=0, vmax=2e-3)
    else:                # fora da diagonal: tem sinal -> colormap divergente
        ax.imshow(sl, cmap="RdBu_r", origin="lower", vmin=-5e-4, vmax=5e-4)
    ax.set_title(nomes[k])
    ax.axis("off")

axes.flat[6].imshow(md_from_channels(comp)[100].T, cmap="gray", origin="lower")
axes.flat[6].set_title("MD = traco/3")
axes.flat[6].axis("off")
axes.flat[7].imshow(fa_from_channels(comp)[100].T, cmap="gray", origin="lower",
                    vmin=0, vmax=1)
axes.flat[7].set_title("FA (das 6 componentes)")
axes.flat[7].axis("off")
plt.tight_layout()
plt.show()

print(f"componentes negativas: {np.mean(comp < 0) * 100:.1f}% "
      "-> clipar em a_min=0 destruiria informacao real")


# **Dados**

O primeiro passo de um projeto é **olhar para os dados**: entender onde estão os arquivos,
como estão organizados e separar sujeitos para treino, validação e teste.

O split já foi feito por pacientes, presente nos arquivos `train.json`, `val.json` e
`test.json` em `data/` (listas de IDs de sujeitos).


## Classe 3D: `BrainHack3Data`

Cada sujeito fica em uma subpasta de `data/HCP_dataset_brainhack_2026`.
Usamos o mapa **FA** como imagem de entrada e a máscara **CC** (MRIcloud) como target.

Organizamos o código de forma **orientada a objetos (OO)**: a classe
`BrainHack3Data` tem uma única responsabilidade:
**ler** os volumes 3D do disco. O pré-processamento fica em **transformadas** opcionais,
passadas no construtor.

> `BrainHack3Data` carrega FA e CC de cada sujeito e devolve os volumes 3D.
> Se `transform` for informado, chama `transform(img, mask)`.


In [ ]:
# TAREFA: como saber o tamanho do split de validação? Dica: inicialize primeiro o dataset de validação!

# Código do dataset 3D

In [ ]:
# NOTA: era `from brainhack import DATA_JSON`. O módulo fixa DATA_JSON no
# caminho do Kaggle, o que sobrescreveria o valor resolvido na célula de
# configuração e quebraria a execução local.
from torch.utils.data import Dataset
from monai.transforms import Orientation
from monai.data import MetaTensor


class BrainHack3Data(Dataset):
    '''
    Dataset que acessar arquivos 3D dos pacientes, seguindo o split de treino, validação ou teste (mode).

    Separar dados no nível do paciente é essencial para evitar contaminação (dados do mesmo paciente no treino e teste por exemplo).

    A entrada pode ser a FA (1 canal) ou o tensor de difusão D (6 canais),
    conforme `input_mode`.
    '''
    # Uso de propriedades da classe para constantes relacionadas ao dataset.
    FA_FILE = "FA.nii"
    EVALS_FILE = "evals.nii"
    EVECS_FILE = "evecs.nii"
    CC_FILE = "cc_mask_mricloud_1.25.nii"


    def __init__(self, mode: str, transform=None, fix=True, input_mode=INPUT_MODE):

        # Biblioteca PATH permite construção dinâmica de caminhos com operador /
        # e train/val/test carregavam o mesmo indice -> montamos o nome do JSON a partir de mode.
        split_path = DATA_JSON / f"{mode}.json"

        # Salvamos o índice de sujeitos para o mode informado.
        with open(split_path) as f:
            self.subject_ids = json.load(f)

        # Salvamos o objeto transformada, opcional.
        self.transform = transform
        self.fix = fix
        self.input_mode = input_mode

        if self.fix:
            print("Aplicando fix.")
            self.las = Orientation(axcodes="LAS")

    def __len__(self):
        # Na abstração de orientação a objetos do Python, o __len__ roda quando a função len() é chamada sobre o objeto.
        return len(self.subject_ids)

    def _load_input(self, subject_dir):
        '''Volume de entrada no formato [canal, X, Y, Z].'''
        if self.input_mode == "fa":
            fa = nib.load(subject_dir / BrainHack3Data.FA_FILE).get_fdata(dtype=np.float32)
            return np.expand_dims(fa, 0)

        # Tensor: reconstruido da autodecomposicao (mesma conta da celula 8).
        evals = nib.load(subject_dir / BrainHack3Data.EVALS_FILE).get_fdata(dtype=np.float32)
        evecs = nib.load(subject_dir / BrainHack3Data.EVECS_FILE).get_fdata(dtype=np.float32)
        if evecs.shape[-1] == 9:
            evecs = evecs.reshape(*evecs.shape[:-1], 3, 3)
        D = np.einsum('...ij,...j,...kj->...ik', evecs, evals, evecs)

        # (X, Y, Z, 6) -> (6, X, Y, Z). O nan_to_num cobre voxels onde o ajuste
        # do tensor nao convergiu (fora do cerebro).
        comp = np.nan_to_num(tensor_to_channels(D))
        return np.ascontiguousarray(np.moveaxis(comp, -1, 0), dtype=np.float32)

    def __getitem__(self, i):
        subject_id = self.subject_ids[i]
        subject_dir = DATA_DIR / subject_id

        cc_nii = nib.load(subject_dir / BrainHack3Data.CC_FILE)

        img = self._load_input(subject_dir)
        mask = np.expand_dims(cc_nii.get_fdata(), 0).astype(np.float32)

        if self.fix:
            mask = self.las(MetaTensor(mask, affine=cc_nii.affine)).numpy()

        if self.transform is not None:
            img, mask = self.transform(img, mask)

        metadata = {"subject_id": subject_id}
        return img, mask, metadata


## Transformadas do dataset 3D

Separamos a **leitura** do **pré-processamento** usando objetos chamáveis.
Cada objeto **transformada** recebe `img, mask` e devolve `img, mask`.

Neste tutorial usamos duas transformadas (nessa ordem):
1. **`MinMaxNormalize`**: normaliza a intensidade do FA para valores entre [0, 1]
2. **`ExtractMidSagittalSlice`**: extrai a fatia sagital média do volume 3D

`ComposeTransforms` encadeia as etapas, repassando `x, y` de uma transformada para a próxima.

Note que esse design pode ser utilizado tanto para pré-processamento quanto aumentação de dados em tempo real.


In [ ]:
class ComposeTransforms:
    """Aplica uma lista de transformadas em sequência."""
    def __init__(self, transforms):
        # Note que transforms é uma lista de objetos "chamáveis" (funções que implementam __call__).
        self.transforms = transforms

    def __call__(self, x, y=None):

        # Aplica cada transformada em sequência sobre tuplas x, y.
        # y=None é o caminho de inferência (volume novo, sem máscara).
        for t in self.transforms:
            x, y = t(x, y)

        return x, y


class MinMaxNormalize:
    """Normaliza a FA para o intervalo [0, 1]."""
    def __call__(self, x, y=None):
        xmin, xmax = x.min(), x.max()
        if xmax > xmin:
            x = (x - xmin) / (xmax - xmin)
        return x, y


class ScaleDiffusivity:
    """Normaliza o tensor por uma difusividade de referência FIXA.

    Por que não min-max, como na FA:

    * as componentes fora da diagonal são negativas (~6% dos valores). Qualquer
      normalização que corte em zero descarta metade da informação de
      orientação -- justamente o motivo de usar D em vez da FA.
    * min-max é por sujeito: um único voxel ruidoso reescala o volume inteiro, e
      a mesma difusividade vira números diferentes em sujeitos diferentes. Uma
      escala fixa mantém os sujeitos comparáveis e mantém o zero significando
      "sem difusão".
    """
    def __init__(self, d_ref=D_REF):
        self.d_ref = d_ref

    def __call__(self, x, y=None):
        # O clip é só rede de segurança: afeta <0.01% dos valores no dataset.
        return np.clip(x / self.d_ref, -1.0, 1.0).astype(np.float32), y


class ExtractMidSagittalSlice:
    """
    Extrai a fatia sagital média de volumes 3D.
    Critério: menor FA médio entre as fatias com tecido suficiente segundo a
    máscara de CÉREBRO (arquivo T1_brain_mask_1.25.nii* do sujeito ou, na falta
    dele, fa_vol > 0). A máscara do corpo caloso (o alvo y) NÃO é usada aqui:
    ela é o que a rede deve prever, e num volume novo ela nem existe.

    Com o tensor na entrada não existe um canal de FA para usar no critério,
    então a FA é recalculada das 6 componentes -- assim a fatia escolhida é a
    MESMA que seria escolhida com a FA na entrada.
    """
    def __init__(self, sagittal_axis=0):
        self.sagittal_axis = sagittal_axis

    def _criterion_map(self, x):
        # x é [canal, X, Y, Z]; devolve um volume escalar de FA.
        if x.shape[0] == 1:
            return x[0]
        return fa_from_channels(np.moveaxis(x, 0, -1))

    def _brain_mask(self, fa_vol, subject_dir=None):
        # Máscara de cérebro: vem da ENTRADA, nunca do alvo.
        if subject_dir is not None:
            hits = sorted(Path(subject_dir).glob("T1_brain_mask_1.25.nii*"))
            if hits:
                return load_nifti(hits[0]) > 0
        # Sem arquivo: aproximação a partir da própria FA (voxels com sinal).
        return fa_vol > 0


    def _find_slice_index(self, fa_vol):
        other_axes = tuple(i for i in range(fa_vol.ndim) if i != self.sagittal_axis)
        mask_count = self._brain_mask(fa_vol).sum(axis=other_axes)
        fa_mean = fa_vol.mean(axis=other_axes)
        fa_mean[mask_count <= 0.90 * mask_count.max()] = 1
        return int(np.argmin(fa_mean))

    def __call__(self, x, y=None):
        fa_vol = self._criterion_map(x)
        slice_idx = self._find_slice_index(fa_vol)  # só depende da entrada
        # +1 porque o eixo 0 de x é o de canais: recorta TODOS os canais.
        x = np.take(x, slice_idx, axis=self.sagittal_axis + 1).astype(np.float32)
        if y is None:
            return x, None  # inferência: não há máscara a recortar
        cc_slice = np.take(y[0], slice_idx, axis=self.sagittal_axis)
        y = np.expand_dims((cc_slice > 0).astype(np.float32), 0)
        return x, y


normalize = MinMaxNormalize() if INPUT_MODE == "fa" else ScaleDiffusivity()

preprocess_3d = ComposeTransforms([
    normalize,
    ExtractMidSagittalSlice(sagittal_axis=SAGITTAL_AXIS)
])


In [ ]:
class BrainHack3Data2D(Dataset):
    '''
    Esse dataset modela nossa tarefa bidimensional: segmentar o corpo caloso de uma fatia sagital média usando uma UNet.

    A entrada pode ter 1 canal (FA) ou 6 canais (componentes de D) -- o código
    abaixo é o mesmo nos dois casos.
    '''
    def __init__(self, mode, transform=None):
        '''
        mode: train, val ou test
        transform: transformação opcional a ser aplicada aos dados
        '''

        # Indexa paths dos arquivos que pré-processamos acima, simplesmente guardando todos os arquivos com fim .npz em uma lista.
        # CUIDADO: a ordem de "glob" não é determinística! Por isso o sorted.
        self.dataset = sorted(glob(os.path.join(PROCESSED_DATA_FOLDER, mode, "*.npz")))
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, i):
        '''
        Lê o arquivo .npz e prepara os tensores PyTorch para o treino.
        '''
        npz = np.load(self.dataset[i])
        # A imagem normalizada é float (com o tensor, inclusive negativa):
        # ler como uint8 zeraria tudo. A máscara continua binária.
        img = npz["img"].astype(np.float32)          # [canal, altura, largura]
        tgt = npz["tgt"].astype(np.uint8).squeeze()

        # O Albumentations trabalha em HWC; nossos .npz estão em CHW.
        img = np.moveaxis(img, 0, -1)                # (H, W, C)

        if self.transform is not None:
            out = self.transform(image=img, mask=tgt)
            img, tgt = out["image"], out["mask"]

        # Formato esperado pela rede: [canal, altura, largura]
        img = torch.from_numpy(np.ascontiguousarray(np.moveaxis(img, -1, 0))).float()
        tgt = torch.from_numpy(tgt).float().unsqueeze(0)
        return img, tgt


In [ ]:
# Teste: fornecer transformada no construtor do dataset 3D
brain_hack_ds = BrainHack3Data("train", transform=preprocess_3d)

# A transformada deve funcionar em qualquer amostra.
random.choice(brain_hack_ds)

Aplicando fix.


/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.array Orientation.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


(array([[[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]]], dtype=float32),
 array([[[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]]], dtype=float32),
 {'subject_id': '765864'})

## Debugging dos dados brutos

Antes de treinar, é importante **visualizar** DENOVO a entrada e máscara após a aplicação das transformadas.
Confira shape, intensidades e se a fatia sagital faz sentido anatomicamente.


In [ ]:
debug_dataset = BrainHack3Data("train", transform=preprocess_3d)
img, mask, metadata = debug_dataset[random.randrange(len(debug_dataset))]

print(f"Sujeito: {metadata['subject_id']}")
print(f"Entrada ({INPUT_MODE}) shape: {img.shape}  CC shape: {mask.shape}")
print(f"Entrada min/max: {img.min():.3f} / {img.max():.3f}")
if img.shape[0] == N_TENSOR_CHANNELS:
    print(f"Fracao negativa: {np.mean(img < 0) * 100:.1f}% (esperado: so fora da diagonal)")

# TAREFA: Use seu código de visualização para verificar se as transformadas funcionaram.


## Pré-processamento para segmentação na fatia mid-sagittal (2D)

A função preprocess extrai **uma fatia sagital média por volume** e salva em `.npz`.

Salvar em disco acelera o treino: o código vai ler fatias 2D já normalizadas e prontas para o treino em vez de um volume NIfTI original.


In [ ]:
from tqdm import tqdm


def preprocess(preprocess_fn):
    '''
    Essa função faz o pré-processamento dos dados usando a transformada preprocess_fn e salva em disco.
    Uma chamada dessa função deve gerar os dados 2D de treino, validação e teste.

    A modularização anterior de abstração do dataset 3D, e transformada como objeto chamável deixa esse código bem simples.

    NOTA: PROCESSED_DATA_FOLDER vem da célula de configuração (depende de
    INPUT_MODE), e não do módulo brainhack -- senão as fatias de FA e as de
    tensor cairiam na mesma pasta com os mesmos nomes.
    '''
    for mode in ["train", "val", "test"]:
        out_dir = Path(PROCESSED_DATA_FOLDER) / mode
        out_dir.mkdir(parents=True, exist_ok=True)

        dataset = BrainHack3Data(mode, transform=preprocess_fn)

        for img, tgt, metadata in tqdm(dataset, desc=f"Preprocess {mode}"):
            subject_id = metadata["subject_id"]
            save_path = out_dir / f"{subject_id}.npz"
            np.savez_compressed(save_path, img=img, tgt=tgt)

# Rodar pré-processamento para salvar fatias 2D, usando a transformada definida lá em cima.
preprocess(preprocess_3d)


## Dataset 2D para treino: `BrainHack3Data2D`

Esta classe lê os `.npz` pré-processados e entrega tensores *PyTorch* para a UNet.


In [ ]:
# NOTA: aqui o notebook original fazia `from brainhack import BrainHack3Data2D`.
# A versão do módulo assume 1 canal (faz squeeze na imagem e unsqueeze no
# final), então sobrescreveria a classe multi-canal definida acima e o treino
# quebraria com 6 canais. Seguimos com a definição do notebook.
print(BrainHack3Data2D)


### Data augmentation

Exemplo de aumentação de dados com a biblioteca **Albumentations**.

No treino, pertubamos os dados com rotação leve e recorte aleatório de patches 64×64. Detalhes estão dentro da função (no outro notebook).

In [ ]:
class RotateCropTensor:
    '''
    Rotação + crop aleatório consistentes com um campo tensorial.

    Por que não usar A.Rotate direto: o Albumentations gira o GRID, mas não tem
    como saber que os 6 canais são as componentes de um tensor e que elas
    precisam girar junto (D' = R D R^T). Sem isso a rede treina com tensores
    apontando para direções que não existem na imagem -- um campo escalar como
    a FA não tem esse problema, um campo tensorial tem.

    A rotação acontece no plano (y, z) da fatia sagital, então a matriz 3D
    correspondente é uma rotação em torno do eixo x (rotation_matrix_x).
    '''
    def __init__(self, limit_deg=10, crop=64, p=0.5):
        self.limit_deg = limit_deg
        self.crop = crop
        self.p = p

    def __call__(self, image, mask):
        if random.random() < self.p:
            # O ângulo é sorteado AQUI (e não dentro do A.Rotate) para podermos
            # aplicar exatamente a mesma rotação nas componentes do tensor.
            angle = random.uniform(-self.limit_deg, self.limit_deg)
            out = A.Rotate(limit=(angle, angle), p=1.0)(image=image, mask=mask)
            image, mask = out["image"], out["mask"]
            if image.shape[-1] == N_TENSOR_CHANNELS:
                # Verificado empiricamente: A.Rotate(+a) move o conteúdo por R.
                # (O Rotate do MONAI, por comparação, move por R^T -- a
                # convenção não é a mesma entre bibliotecas, então confira
                # antes de reusar isto em outro lugar.)
                image = rotate_tensor_channels(
                    image, rotation_matrix_x(np.deg2rad(angle))
                )
        return A.RandomCrop(width=self.crop, height=self.crop, p=1.0)(
            image=image, mask=mask
        )


def get_transform(transform_str: str):
    '''
    Factory de transformações (None = sem augmentation).

    A ideia de uma "fábrica de objetos" é muito útil para organizar código de aprendizado profundo.

    Simplesmente a string de entrada controla qual o objeto de transformada que será instanciado, facilitando reproducibilidade de experimentos.

    transform_str deve ser guardado como um hiperparâmetro do experimento.

    Note que a biblioteca Albumentations de transformadas 2D segue um esquema de composição semelhante a nossas transformadas 3D!
    '''
    if transform_str == "rotate_crop":
        # NOTA: era A.Compose([A.Rotate(...), A.RandomCrop(...)]). Virou uma
        # classe própria porque a rotação precisa girar também as componentes.
        return RotateCropTensor(limit_deg=10, crop=64, p=0.5)
    if transform_str == "center_crop":
        return A.Compose([
            A.CenterCrop(width=128, height=128),
        ])
    return None


## Debug do Dataset 2D

Compare uma amostra **sem** e **com** augmentation para entender o que a rede verá no treino.

Esse dataset 2D já retorna tensores do PyTorch!


In [ ]:
# Sem augmentation
train_ds = BrainHack3Data2D("train")
img, tgt = train_ds[random.randrange(len(train_ds))]
print(img.shape, tgt.shape)

# Com augmentation (patch 64x64)
aug = get_transform("rotate_crop")
train_aug = BrainHack3Data2D("train", transform=aug)
img, tgt = train_aug[random.randrange(len(train_aug))]
print(img.shape, tgt.shape)

# Validação com center crop (128x128)
val_ds = BrainHack3Data2D("val", transform=get_transform("center_crop"))
img, tgt = val_ds[random.randrange(len(val_ds))]
print(img.shape, tgt.shape)


torch.Size([1, 174, 145]) torch.Size([1, 174, 145])
torch.Size([1, 64, 64]) torch.Size([1, 64, 64])
torch.Size([1, 128, 128]) torch.Size([1, 128, 128])


In [ ]:
# Carrega todos os itens antes do treino para detectar erros cedo.
# verifique se o número de amostrar é o esperado. Se não, ainda há algum erro acima.
for mode in ["train", "val", "test"]:
    _ = [x for x in BrainHack3Data2D(mode)]
    print(f"{mode}: ok ({len(BrainHack3Data2D(mode))} amostras)")


train: ok (115 amostras)
val: ok (14 amostras)
test: ok (15 amostras)


# **Treinamento**


### Hiperparâmetros

Dicionário com os hiperparâmetros do experimento. 

Muito importante organizar os hiperparâmetros do seu experimento. Evita "números mágicos" espalhados no código.


In [ ]:
LOGS_ROOT = "logs"

hparams = {
    "experiment_name": f"BrainhackCC_{INPUT_MODE}",
    "train_transform_str": "rotate_crop",
    "eval_transform_str": "center_crop",
    "max_epochs": 20,
    "batch_size": 10,
    "nworkers": 0,
    # nin acompanha a entrada escolhida: 1 (FA) ou 6 (componentes de D).
    "nin": N_IN,
    "nout": 1,
    "lr": 1e-4,
    "precision": 32,
    "debug": False
}

hparams["experiment_dir"] = os.path.join(LOGS_ROOT, hparams["experiment_name"])

for k, v in hparams.items():
    print(f"{k}: {v}")


## DataModule e DataLoaders

O `LightningDataModule` centraliza Datasets e  DataLoaders.


In [ ]:
class BrainHack3DataModule(pl.LightningDataModule):
    '''
    O DataModule ajuda a organizar o seu experimento, especialmente com a seção setup, onde 
    você pode centrar tarefas de criação do dataset. 

    Aqui no nosso exemplo simples, o LightningDataModule inicializa os datasets e seus respectivos DataLoaders.

    DataLoaders são responsáveis por iterar sobre os dados de forma eficiente e criar batches (conjuntos de amostras).
    
    Redes convolucionais como a UNet geralmente processam batches inteiros de uma vez, em vez de aprender de amostra em amostra.
    '''
    def __init__(self, hparams):
        super().__init__()
        self.save_hyperparameters(hparams)

    def setup(self, stage=None):
        train_t = get_transform(self.hparams.train_transform_str)
        eval_t = get_transform(self.hparams.eval_transform_str)
        self.train = BrainHack3Data2D("train", transform=train_t)
        self.val = BrainHack3Data2D("val", transform=eval_t)
        self.test = BrainHack3Data2D("test", transform=eval_t)

    def train_dataloader(self):
        return DataLoader(self.train, batch_size=self.hparams.batch_size,
                          num_workers=self.hparams.nworkers, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val, batch_size=self.hparams.batch_size,
                          num_workers=self.hparams.nworkers, shuffle=False)

    def test_dataloader(self):
        return DataLoader(self.test, batch_size=self.hparams.batch_size,
                          num_workers=self.hparams.nworkers, shuffle=False)

In [ ]:
# Teste de um dataloader
data_module = BrainHack3DataModule(hparams)
data_module.setup()
train_loader = data_module.train_dataloader()

# Verifica se o DataLoader está funcionando.
# O dataloader retorna batches! A rede processa grupos de imagens e máscaras.
batch = next(iter(train_loader))
print(batch[0].shape, batch[1].shape)

torch.Size([10, 1, 64, 64]) torch.Size([10, 1, 64, 64])


In [ ]:
# Visualizar o batch com grid da biblioteca torchvision
images, labels = batch[0], batch[1]

# image_grid = torchvision.utils.make_grid(images, nrow=4).cpu().numpy()
# label_grid = torchvision.utils.make_grid(labels, nrow=4).cpu().numpy()

# plt.figure(figsize=(12, 6))
# plt.subplot(1, 2, 1)
# plt.imshow(image_grid[0])
# plt.subplot(1, 2, 2)
# plt.imshow(label_grid[0])
# plt.show()

## Função de perda: Dice Loss

$$ D = \frac{2 \sum_i p_i t_i}{\sum_i p_i + \sum_i t_i} $$

Usamos `1 - Dice` como função de perda, comum em segmentação com classes desbalanceadas, onde p é a predição e t o alvo (*target*).
O objetivo da função de perda é expressar o quando o modelo está errando, servindo como objetivo de minimização, e guiando todo o treino.


In [ ]:
def dice_coeff(input: Tensor, target: Tensor, reduce_batch_first: bool = False, epsilon=1e-6):
    '''
    Calcula o coeficiente de Dice entre a entrada e o alvo.

    Ilustração: 2x overlap / união entre as máscaras.

    Note um detalhe importante: durante o treino, temos um conjunto de imagens e os alvos (máscaras) respectivas.
    Podemos calcular o Dice para cada imagem e depois calcular a média, ou considerar o batch inteiro de uma vez.
    '''
    assert input.size() == target.size()
    if input.dim() == 2 and reduce_batch_first:
        raise ValueError(f"Dice: tensor sem batch (shape {input.shape})")

    # Se a entrada tem duas dimensões ou queremos tratar o batch inteiro de uma vez, aplicamos o dice na linearização dos valores. 
    if input.dim() == 2 or reduce_batch_first:
        inter = torch.dot(input.reshape(-1), target.reshape(-1))
        sets_sum = torch.sum(input) + torch.sum(target)
        if sets_sum.item() == 0:
            sets_sum = 2 * inter
        return (2 * inter + epsilon) / (sets_sum + epsilon)

    # Caso a entrada tenha mais de duas dimensões, calculamos o Dice para cada elemento do batch.
    dice = 0
    for i in range(input.shape[0]):
        dice += dice_coeff(input[i, ...], target[i, ...])

    return dice / input.shape[0]

## Arquitetura: UNet

UNet para segmentação 2D ou 3D. Em vez de importar de bibliotecas, vamos usar um código explícito simplificado de implementação da UNet com PyTorch.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, norm, reduce, dim):
        '''
        Módulo dinâmico 3D ou 2D, cria uma camada de duas convoluções com batch normalization e leaky ReLU.

        O Argumento reduce controla o stride da segunda convolução para reduzir a resolução espacial da saída.
        '''
        super().__init__()
        if norm:
            norms = [getattr(nn, f"BatchNorm{dim}")(out_ch) for _ in range(2)]
        else:
            norms = [nn.Identity(), nn.Identity()]

        self.conv = nn.Sequential(
            getattr(nn, f"Conv{dim}")(in_ch, out_ch, kernel_size=3, padding=1, stride=1, bias=False),
            norms[0],
            nn.LeakyReLU(inplace=True),
            getattr(nn, f"Conv{dim}")(out_ch, out_ch, kernel_size=3, padding=1, stride=2 if reduce else 1, bias=False),
            norms[1],
            nn.LeakyReLU(inplace=True),
        )
        self.residual_connection = getattr(nn, f"Conv{dim}")(
            in_ch, out_ch, kernel_size=1, padding=0, stride=2 if reduce else 1, bias=False
        )

    def forward(self, x):
        return self.conv(x) + self.residual_connection(x)


class Up(nn.Module):
    def __init__(self, in_ch, out_ch, norm, dim):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, align_corners=True, mode="bilinear")
        self.conv = DoubleConv(in_ch, out_ch, norm, reduce=False, dim=dim)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        # Ajuste de tamanho quando as dimensões não batem após upsample
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, (diffY // 2, diffY - diffY // 2, diffX // 2, diffX - diffX // 2))
        return self.conv(torch.cat([x2, x1], dim=1))


class UNetEncoder(nn.Module):
    def __init__(self, n_channels, init_channel, norm, dim):
        super().__init__()
        self.inc = DoubleConv(n_channels, init_channel, norm=norm, reduce=False, dim=dim)
        self.down1 = DoubleConv(init_channel, init_channel * 2, norm=norm, reduce=True, dim=dim)
        self.down2 = DoubleConv(init_channel * 2, init_channel * 4, norm=norm, reduce=True, dim=dim)
        self.down3 = DoubleConv(init_channel * 4, init_channel * 8, norm=norm, reduce=True, dim=dim)
        self.down4 = DoubleConv(init_channel * 8, init_channel * 8, norm=norm, reduce=True, dim=dim)

    def forward(self, x):
        out_1 = self.inc(x)
        out_2 = self.down1(out_1)
        out_3 = self.down2(out_2)
        out_4 = self.down3(out_3)
        y = self.down4(out_4)
        return y, out_1, out_2, out_3, out_4


class UNetDecoder(nn.Module):
    def __init__(self, n_classes, init_channel, norm, dim):
        super().__init__()
        self.up1 = Up(16 * init_channel, 4 * init_channel, norm, dim=dim)
        self.up2 = Up(8 * init_channel, 2 * init_channel, norm, dim=dim)
        self.up3 = Up(4 * init_channel, init_channel, norm, dim=dim)
        self.up4 = Up(2 * init_channel, init_channel, norm, dim=dim)
        self.outc = getattr(nn, f"Conv{dim}")(init_channel, n_classes, kernel_size=1, bias=False)

    def forward(self, y, out_1, out_2, out_3, out_4):
        y = self.up1(y, out_4)
        y = self.up2(y, out_3)
        y = self.up3(y, out_2)
        y = self.up4(y, out_1)
        return self.outc(y)


class UNet(nn.Module):
    def __init__(self, n_channels, n_classes, norm, dim, init_channel):
        super().__init__()
        self.enc = UNetEncoder(n_channels, init_channel, norm, dim)
        self.dec = UNetDecoder(n_classes, init_channel, norm, dim)
        print(f"UNet: in={n_channels} out={n_classes} dim={dim} init_ch={init_channel}")

    def forward(self, x):
        return self.dec(*self.enc(x))

## Lightning Module

O Lightning Module é uma abstração da arquitetura por completo, implementando os passos de treino e validação (o que acontece com um batch).

O Module consome batches do DataLoader. O lightning resolve questões de otimização, como aplicação do otimizador e mover os tensores para a GPU se for o caso.


In [ ]:
class CCSegmentation(pl.LightningModule):
    def __init__(self, hparams):
        '''
        Lightning Module é uma abstração da arquitetura por completo, implementando os passos de treino e validação (o que acontece com um batch).

        No construtor, inicializamos a arquitetura, e salvamos os hiperparâmetros, que agora podem ser referenciados como `self.hparams`.
        '''
        super().__init__()
        self.save_hyperparameters(hparams)
        self.model = UNet(
            n_channels=self.hparams.nin,
            n_classes=self.hparams.nout,
            norm=True,
            dim="2d",
            init_channel=32,
        )

    def forward(self, x):
        return self.model(x).sigmoid()

    def step(self, mode, batch):
        '''
        O Lightning Module implementa o passo de treino e validação.

        O método `step` é chamado para cada batch, e implementa o que deve ser feito com ele.

        Basicamente, precisamos passar os dados de entrada pela rede e calcular a perda.

        O Lightning cuida de gerenciamento do otimizador, mover para GPU, etc.
        '''
        x, y = batch

        y_hat = self.forward(x)

        loss = 1 - dice_coeff(y_hat, y)
        
        if mode == "train":
            self.log("loss", loss, on_epoch=True, on_step=True)
            return loss
        elif mode == "val":
            self.log("val_loss", loss, on_epoch=True, on_step=False, prog_bar=True)

    def training_step(self, batch, batch_idx):
        return self.step("train", batch)

    def validation_step(self, batch, batch_idx):
        self.step("val", batch)

    def configure_optimizers(self):
        '''
        Aqui configuramos o otimizador.
        '''
        return Adam(self.model.parameters(), lr=self.hparams.lr)

## Treinar

O `Trainer` do Lightning gerencia o loop de épocas, GPU/CPU, checkpoints e métricas do TensorBoard.
Tudo fica em `logs/<experiment_name>/` (definido em `hparams["experiment_name"]`).

Para acompanhar o treino no TensorBoard:

```python
%tensorboard --logdir logs
```


In [ ]:
# Lightning Module e DataModule são os dois objetos necessários para o treinamento.
model = CCSegmentation(hparams)
data = BrainHack3DataModule(hparams)

experiment_dir = hparams["experiment_dir"]
os.makedirs(experiment_dir, exist_ok=True)

# Logger do PyTorch Lightning que salva métricas do experimento no TensorBoard.
logger = TensorBoardLogger(save_dir=LOGS_ROOT, 
                           name=hparams["experiment_name"])

# Callback do PyTorch Lightning que salva o modelo com o menor loss de validação.
checkpoint_callback = ModelCheckpoint(
    dirpath=experiment_dir,
    filename="{epoch}-{val_loss:.2f}",
    monitor="val_loss",
    mode="min",
)

# Trainer do PyTorch Lightning gerencia o loop de épocas, GPU/CPU, checkpoints e métricas do TensorBoard.
trainer = pl.Trainer(
    max_epochs=hparams["max_epochs"],
    devices=1,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    precision=hparams["precision"],
    fast_dev_run=hparams["debug"],
    logger=logger,
    default_root_dir=experiment_dir,
    callbacks=[checkpoint_callback],
    log_every_n_steps=1
)

# Roda o treinamento
trainer.fit(model, data)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


UNet: in=1 out=1 dim=2d init_ch=32


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /kaggle/working/logs/BrainhackCC exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ UNet │  3.5 M │ train │     0 │
└───┴───────┴──────┴────────┴───────┴───────┘

Trainable params: 3.5 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 3.5 M                                                                                                
Total estimated model params size (MB): 14.185                                                                     
Modules in train mode: 93                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=20` reached.


# **Análise de predição**

Este código carrega o melhor checkpoint e rode inferência em itens do conjunto de **validação**
(com o mesmo `center_crop` da avaliação). Compara-se visualmente FA, alvo e predição contínua.


In [ ]:
# Carregando o checkpoint
ckpt = sorted(glob(os.path.join(hparams["experiment_dir"], "*.ckpt")))[-1]
print(f"Checkpoint: {ckpt}")

# Carrega o modelo. modo Eval desabilita partes do modelo que não devem rodar fora do treino como Dropout.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trained = CCSegmentation.load_from_checkpoint(ckpt).eval().to(device)

# Inicializa fatias de val.
val_ds = BrainHack3Data2D("val", transform=get_transform("center_crop"))


def to_display(img):
    """Mapa escalar para a figura: a própria FA, ou a FA recalculada de D.

    Com 6 canais não existe "a imagem" para mostrar -- img.squeeze() devolveria
    (6, H, W) e quebraria o imshow.
    """
    arr = img.numpy()
    if arr.shape[0] == 1:
        return arr[0]
    return fa_from_channels(np.moveaxis(arr, 0, -1))


# Computa saídas do modelo treinado para o conjunto de val
imgs_np, tgts_np, preds_np = [], [], []
for i in range(len(val_ds)):
    img, tgt = val_ds[i]

    # IMPORTANTE: torch.no_grad() desabilita computação de gradientes (necessários somente para treino)
    # economiza memória e tempo de execução.
    with torch.no_grad():
        pred = trained(img.unsqueeze(0).to(device)).cpu().squeeze().numpy()
        # Porque o .cpu, .squeeze e .numpy são necessários?

    # Guarda entradas e saidas para plotar depois.
    imgs_np.append(to_display(img))
    tgts_np.append(tgt.squeeze().numpy())
    preds_np.append(pred)

    # Visualização de predições
    plt.figure(figsize=(12, 4))
    for j, (arr, title) in enumerate([
        (imgs_np[-1], f"Entrada: FA ({INPUT_MODE}), val {i}"),
        (tgts_np[-1], "CC (alvo)"),
        (preds_np[-1], "Predição (contínua)"),
    ]):
        plt.subplot(1, 3, j + 1)
        plt.imshow(arr, cmap="gray")
        plt.title(title)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

    if i==5:
        break

print(f"{len(val_ds)} amostras de val inferidas.")


## Computação de métricas

Função auxiliar que compara máscaras binárias (ground truth vs predição) com **SimpleITK**:
Dice e distância de Hausdorff por estrutura.

As entradas devem ser arrays inteiros (ex.: `uint8` com 0/1). 

Como a saída da rede é contínua, por isso o próximo passo é escolher um limiar de binarização.


In [ ]:
def seg_metrics(gts, preds, metrics, struct_names=["cc"]):
    # Loop sobre pares de predição e alvo para compará-los
    for gt, pred, label in zip(gts, preds, struct_names):
        # sitk implementa filtors que fornecem múltiplas métricas.
        overlap = sitk.LabelOverlapMeasuresImageFilter()
        hausdorff = sitk.HausdorffDistanceImageFilter()

        # converte numpy para formato do sitk
        gt_img = sitk.GetImageFromArray(gt)
        pred_img = sitk.GetImageFromArray(pred)

        # executa filtros (métricas)
        overlap.Execute(gt_img, pred_img)

        # salva métricas em dicionário dinâmico
        metrics[label]["dice"].append(overlap.GetDiceCoefficient())
        metrics[label]["jaccard"].append(overlap.GetJaccardCoefficient())
        try:
            hausdorff.Execute(gt_img, pred_img)
            metrics[label]["hd"].append(hausdorff.GetHausdorffDistance())
        except Exception:
            metrics[label]["hd"].append(nan)

In [ ]:
threshold = 0.5

# metrics_ths é um dicionário de dicionários dinâmicos, que guarda métricas por estrutura
# Porque seria interessante guardar as métricas dessa forma?
metrics_ths = defaultdict(lambda: defaultdict(list))

# Loop sobre pares de predição e alvo para compará-los
for tgt_np, pred in zip(tgts_np, preds_np):
    # Converte para binário com o threshold escolhido
    gt_u8 = (tgt_np > 0).astype(np.uint8)
    pred_u8 = (pred > threshold).astype(np.uint8)

    # Calcula métricas
    seg_metrics(gt_u8[None], pred_u8[None], metrics_ths, struct_names=["cc"])

metrics_ths["cc"]["dice"]

[0.9471264367816091,
 0.945945945945946,
 0.9595854922279792,
 0.9647355163727959,
 0.9497326203208556,
 0.9232954545454546]

## Pergunta: Onde binarizar?

A rede produz valores contínuos em [0, 1]. Para métricas de segmentação precisamos de uma máscara
binária, e o **threshold** importa: muito baixo gera falso positivo; muito alto gera falso negativo.

TAREFA: avalie diversos valores de threshold (o exemplo acima usou 0.5), e descubra qual o melhor para sua rede segundo os dados de validação.


In [ ]:
# TAREFA: descobrir o melhor threshold para a sua rede.


## Avaliação final no teste

Depois de definir o melhor threshold (coloque na variável best_th), agora testamos as métricas no conjunto de **teste**.


In [ ]:
# TAREFA
# Calcule o Dice com seu melhor threshold no split de teste
# Carregando o checkpoint
ckpt = sorted(glob(os.path.join(hparams["experiment_dir"], "*.ckpt")))[-1]
print(f"Checkpoint: {ckpt}")

# Carrega o modelo. modo Eval desabilita partes do modelo que não devem rodar fora do treino como Dropout.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trained = CCSegmentation.load_from_checkpoint(ckpt).eval().to(device)

# Inicializa fatias de test.
test_ds = BrainHack3Data2D("test", transform=get_transform("center_crop"))


def to_display(img):
    """Mapa escalar para a figura: a própria FA, ou a FA recalculada de D.

    Com 6 canais não existe "a imagem" para mostrar -- img.squeeze() devolveria
    (6, H, W) e quebraria o imshow.
    """
    arr = img.numpy()
    if arr.shape[0] == 1:
        return arr[0]
    return fa_from_channels(np.moveaxis(arr, 0, -1))


# Computa saídas do modelo treinado para o conjunto de test
imgs_np, tgts_np, preds_np = [], [], []
for i in range(len(test_ds)):
    img, tgt = test_ds[i]

    # IMPORTANTE: torch.no_grad() desabilita computação de gradientes (necessários somente para treino)
    # economiza memória e tempo de execução.
    with torch.no_grad():
        pred = trained(img.unsqueeze(0).to(device)).cpu().squeeze().numpy()
        # Porque o .cpu, .squeeze e .numpy são necessários?

    # Guarda entradas e saidas para plotar depois.
    imgs_np.append(to_display(img))
    tgts_np.append(tgt.squeeze().numpy())
    preds_np.append(pred)

    # Visualização de predições
    plt.figure(figsize=(12, 4))
    for j, (arr, title) in enumerate([
        (imgs_np[-1], f"Entrada: FA ({INPUT_MODE}), test {i}"),
        (tgts_np[-1], "CC (alvo)"),
        (preds_np[-1], "Predição (contínua)"),
    ]):
        plt.subplot(1, 3, j + 1)
        plt.imshow(arr, cmap="gray")
        plt.title(title)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

    if i==5:
        break

print(f"{len(test_ds)} amostras de test inferidas.")


In [ ]:
threshold = 0.5

# metrics_ths é um dicionário de dicionários dinâmicos, que guarda métricas por estrutura
# Porque seria interessante guardar as métricas dessa forma?
metrics_ths = defaultdict(lambda: defaultdict(list))

# Loop sobre pares de predição e alvo para compará-los
for tgt_np, pred in zip(tgts_np, preds_np):
    # Converte para binário com o threshold escolhido
    gt_u8 = (tgt_np > 0).astype(np.uint8)
    pred_u8 = (pred > threshold).astype(np.uint8)

    # Calcula métricas
    seg_metrics(gt_u8[None], pred_u8[None], metrics_ths, struct_names=["cc"])

metrics_ths["cc"]["dice"]

[0.9504373177842567,
 0.9448051948051949,
 0.9486260454002389,
 0.9382352941176471,
 0.9649595687331537,
 0.9380281690140845]

# DESAFIO: Treinar utilizando T1 ou outros mapas de DTI

## Utilize o notebook Completo, com todo código exposto, para modificar o dataset (ou qualquer outro aspecto do código) para treinar a rede com informações estruturais (T1) e de mapas de DTI.

### DICA: Você pode colocar outros mapas como canais adicionais de entrada na UNet!